# Week 11 — Markov Asset Pricing

**Required reading:** [QuantEcon: Asset Pricing — Finite State Models](https://python.quantecon.org/markov_asset.html)

**Submission:** Complete the notebook, including mathematical explanations in Markdown, Python computations, figures, and economic interpretations. Show derivations rather than only reporting numerical results. Use NumPy and Matplotlib; no external data files are needed.

**Baseline parameters:** $\beta=0.95$, $\gamma=2$, gross consumption/dividend growth factors $g=(0.98,1.02,1.06)$, and

$$P=\begin{pmatrix}0.80&0.15&0.05\\0.10&0.80&0.10\\0.05&0.15&0.80\end{pmatrix}.$$

The growth factor $g_j$ is realized when the **next-period** Markov state is $j$. Unless otherwise stated, prices are measured in units of the current consumption good. All six problems receive equal weight. In matrix calculations, use stable linear solvers and report numerical checks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=8, suppress=True)
beta = 0.95
gamma = 2.0
state_names = ["Recession", "Normal", "Expansion"]
g = np.array([0.98, 1.02, 1.06])
P = np.array([[0.80, 0.15, 0.05],
              [0.10, 0.80, 0.10],
              [0.05, 0.15, 0.80]])
n = len(g)


## Problem 1: 1. Markov Growth Economy

**(a)** Verify that $P$ is a stochastic matrix. **(b)** Compute the invariant distribution $\pi$ satisfying $\pi P=\pi$ and $\sum_i\pi_i=1$. **(c)** Simulate 10,000 states, beginning in the normal state, with a fixed random seed. Compare the empirical frequencies to $\pi$. **(d)** Interpret persistence and the long-run mean gross growth factor $\sum_i\pi_i g_i$.

### Worked derivation and interpretation

The large diagonal elements of $P$ imply substantial state persistence. In a long simulation the relative frequency of each state approaches the invariant probability. The long-run mean gross growth factor is $\pi g$.

In [ ]:
# (a) Row stochasticity
assert np.all(P >= 0) and np.allclose(P.sum(axis=1), 1)
print("Row sums:", P.sum(axis=1))

# (b) Invariant distribution (replace one equation with normalization)
A = P.T - np.eye(n)
A[-1, :] = 1.0
rhs = np.zeros(n); rhs[-1] = 1.0
pi = np.linalg.solve(A, rhs)
print("Stationary distribution:", pi)
print("Stationarity error:", np.max(np.abs(pi @ P - pi)))

# (c) Reproducible simulation
rng = np.random.default_rng(2026)
T = 10_000
states = np.empty(T, dtype=int)
states[0] = 1
for t in range(1, T):
    states[t] = rng.choice(n, p=P[states[t-1]])
freq = np.bincount(states, minlength=n) / T
print("Simulated frequencies:", freq)
print("Stationary mean gross growth:", pi @ g)
plt.figure(figsize=(7, 3.6))
x = np.arange(n)
plt.bar(x-.18, pi, .36, label="Stationary")
plt.bar(x+.18, freq, .36, label="Simulated")
plt.xticks(x, state_names); plt.ylabel("Frequency")
plt.title("Stationary and simulated state frequencies")
plt.legend(); plt.tight_layout(); plt.show()

## Problem 2: 2. Euler Equation and Stochastic Discount Factor

**(a)** Starting from $u'(c_t)=\beta\mathbb E_t[R_{t+1}u'(c_{t+1})]$, derive $1=\mathbb E_t[m_{t+1}R_{t+1}]$ with $m_{t+1}=\beta u'(c_{t+1})/u'(c_t)$. **(b)** Under CRRA utility and $c_{t+1}/c_t=g_j$, derive $m_{ij}=\beta g_j^{-\gamma}$. **(c)** Compute and display the three possible values of the SDF. Explain why low-growth outcomes have larger discount factors.

### Worked derivation and interpretation

The consumption Euler equation $u'(c_t)=\beta\mathbb E_t[R_{t+1}u'(c_{t+1})]$ implies $1=\mathbb E_t[m_{t+1}R_{t+1}]$, where $m_{t+1}=\beta u'(c_{t+1})/u'(c_t)$. With $u'(c)=c^{-\gamma}$ and $c_{t+1}/c_t=g_j$, we have $m_{ij}=\beta g_j^{-\gamma}$. Because marginal utility is high when consumption is low, recession outcomes receive greater state prices per unit probability.

In [ ]:
# CRRA marginal-utility discount factors depend on next-period growth
sdf_next = beta * g**(-gamma)
for name, m in zip(state_names, sdf_next):
    print(f"{name:10s}  m = {m:.8f}")
assert sdf_next[0] > sdf_next[1] > sdf_next[2]

## Problem 3: 3. Lucas Tree and Matrix Pricing

Assume aggregate consumption equals dividends and $d_{t+1}=g_jd_t$ when next period's state is $j$. **(a)** Starting from $p_t=\mathbb E_t[m_{t+1}(p_{t+1}+d_{t+1})]$, show that the state-dependent price-dividend ratio satisfies $v_i=\beta\sum_jP_{ij}g_j^{1-\gamma}(1+v_j)$. **(b)** Define $J_{ij}=P_{ij}g_j^{1-\gamma}$ and derive $(I-\beta J)v=\beta J\mathbf 1$. State the condition ensuring a finite solution. **(c)** Solve with `np.linalg.solve` (not explicit matrix inversion). **(d)** Starting from zero, iterate $v^{(k+1)}=\beta J(\mathbf1+v^{(k)})$ until the sup-norm difference is below $10^{-11}$. Compare results and verify the residual.

### Worked derivation and interpretation

Writing $p_t=v_i d_t$ and $p_{t+1}=v_jd_{t+1}=v_jg_jd_t$, the Euler pricing equation becomes $v_i=\sum_jP_{ij}\beta g_j^{-\gamma}g_j(1+v_j)=\beta\sum_jP_{ij}g_j^{1-\gamma}(1+v_j)$. Thus $(I-\beta J)v=\beta J\mathbf1$. A sufficient and, for this positive irreducible setting, relevant condition for a finite nonnegative perpetuity price is $\rho(\beta J)<1$. Fixed-point iteration converges to the same vector.

In [ ]:
J = P * g[None, :]**(1-gamma)
B = beta * J
rho = max(abs(np.linalg.eigvals(B)))
print("Spectral radius beta*J:", rho)
assert rho < 1, "Finite perpetuity solution requires spectral radius < 1"
v = np.linalg.solve(np.eye(n) - B, B @ np.ones(n))
print("Lucas tree P/D ratios:", v)

v_iter = np.zeros(n)
for k in range(1, 100_001):
    v_next = B @ (np.ones(n) + v_iter)
    if np.max(np.abs(v_next-v_iter)) < 1e-11:
        v_iter = v_next
        break
    v_iter = v_next
else:
    raise RuntimeError("Iteration failed to converge")
print("Iterations:", k)
print("Max difference (solve vs iterate):", np.max(np.abs(v-v_iter)))
print("Bellman/pricing residual:", np.max(np.abs(v - B @ (np.ones(n)+v))))

## Problem 4: 4. Risk Aversion and Persistence

**(a)** For $\gamma\in\{0,1,2,4,6,10\}$, compute the three price-dividend ratios and graph them against $\gamma$. **(b)** Interpret the cross-state and risk-aversion patterns; avoid claiming prices must fall with $\gamma$ in every economy. **(c)** Form $P(\lambda)=\lambda P+(1-\lambda)\mathbf1\pi$, where $\pi$ is the stationary distribution from Problem 1 and $\lambda\in[0,1]$. Verify that every $P(\lambda)$ has invariant distribution $\pi$. Plot price-dividend ratios versus $\lambda$ at $\gamma=2$ and interpret the effect of persistence.

### Worked derivation and interpretation

The effect of $\gamma$ is not a universal monotonicity theorem: changing risk aversion changes discounting of both good and bad growth outcomes. For $\lambda=0$, each row of $P(\lambda)$ equals $\pi$, so state-conditional price-dividend ratios coincide. As $\lambda$ rises, today's state becomes more informative about tomorrow, allowing state-dependent price-dividend ratios to diverge. The invariant distribution remains $\pi$ because $\pi P=\pi$ and $\pi(\mathbf1\pi)=\pi$.

In [ ]:
def lucas_pd(Pmat, growth, beta, risk_aversion):
    Jmat = Pmat * growth[None, :]**(1-risk_aversion)
    Bmat = beta * Jmat
    if max(abs(np.linalg.eigvals(Bmat))) >= 1:
        return np.full(len(growth), np.nan)
    return np.linalg.solve(np.eye(len(growth))-Bmat, Bmat @ np.ones(len(growth)))

gammas = np.array([0., 1., 2., 4., 6., 10.])
pd_by_gamma = np.array([lucas_pd(P, g, beta, z) for z in gammas])
print("Rows correspond to gamma =", gammas)
print(pd_by_gamma)
plt.figure(figsize=(7,4))
for j, name in enumerate(state_names):
    plt.plot(gammas, pd_by_gamma[:,j], 'o-', label=name)
plt.xlabel("Risk aversion gamma"); plt.ylabel("Price / dividend")
plt.title("Lucas tree prices and risk aversion")
plt.legend(); plt.tight_layout(); plt.show()

# P(lambda) interpolates between iid states and original persistent chain.
lambdas = np.linspace(0,1,41)
P_iid = np.tile(pi, (n,1))
assert np.allclose(pi @ P_iid, pi)
pd_by_lambda = []
for lam in lambdas:
    Plam = lam*P + (1-lam)*P_iid
    assert np.allclose(Plam.sum(axis=1), 1)
    assert np.allclose(pi @ Plam, pi)
    pd_by_lambda.append(lucas_pd(Plam, g, beta, 2.0))
pd_by_lambda = np.array(pd_by_lambda)
plt.figure(figsize=(7,4))
for j, name in enumerate(state_names):
    plt.plot(lambdas, pd_by_lambda[:,j], label=name)
plt.xlabel("Persistence lambda"); plt.ylabel("Price / dividend")
plt.title("Persistence with invariant distribution held fixed")
plt.legend(); plt.tight_layout(); plt.show()

## Problem 5: 5. Risk-Free Bonds and Consols

**(a)** A one-period risk-free bond pays one unit next period. Show that its price in state $i$ is $q_i=\beta\sum_jP_{ij}g_j^{-\gamma}$ and its net yield is $r_{f,i}=q_i^{-1}-1$. Compute both. **(b)** A consol pays one unit of the consumption good each period starting next period. Show that $b_i=\beta\sum_jP_{ij}g_j^{-\gamma}(1+b_j)$. Define $H_{ij}=P_{ij}g_j^{-\gamma}$ and solve the linear system. **(c)** Verify both pricing equations numerically and explain the distinction between the consol price and the Lucas-tree price-dividend ratio.

### Worked derivation and interpretation

A one-period claim to one unit has $q_i=\mathbb E_i[m_{t+1}]=\beta\sum_jP_{ij}g_j^{-\gamma}$. The risk-free gross return is $1/q_i$. The consol's recursion is $b_i=\beta\sum_jP_{ij}g_j^{-\gamma}(1+b_j)$. The Lucas tree has a *growing* dividend $d_{t+1}=g_jd_t$, which changes the pricing kernel for the ratio to $g_j^{1-\gamma}$; the consol's coupon is fixed at one unit, so its kernel is $g_j^{-\gamma}$.

In [ ]:
H = P * g[None, :]**(-gamma)
C = beta * H
q = C @ np.ones(n)
rf = 1/q - 1
print("One-period bond prices:", q)
print("Risk-free net yields:", rf)
assert max(abs(np.linalg.eigvals(C))) < 1
b = np.linalg.solve(np.eye(n)-C, C @ np.ones(n))
print("Consol prices:", b)
print("Bond pricing error:", np.max(np.abs(q-C@np.ones(n))))
print("Consol pricing error:", np.max(np.abs(b-C@(np.ones(n)+b))))
plt.figure(figsize=(7,3.6))
plt.bar(state_names, rf*100)
plt.ylabel("Risk-free yield (%)"); plt.title("State-dependent one-period yields")
plt.tight_layout(); plt.show()

## Problem 6: 6. Log Utility and Synthesis

**(a)** Prove analytically that when $\gamma=1$, $v_i=\beta/(1-\beta)$ for every state, provided the relevant transversality condition holds. **(b)** Verify numerically and report the maximum absolute error. **(c)** Show that the Lucas-tree pricing matrix is $J=P$ under log utility. **(d)** Explain in a paragraph how the Euler equation, Markov transition matrix, and linear algebra fit together to determine asset prices. **(e)** Optional: explain what changes if dividends are not identical to aggregate consumption.

### Worked derivation and interpretation

For $\gamma=1$, $J_{ij}=P_{ij}g_j^0=P_{ij}$, hence $J=P$ and $P\mathbf1=\mathbf1$. Guess $v=\bar v\mathbf1$. Then $\bar v\mathbf1=\beta P(\mathbf1+\bar v\mathbf1)=\beta(1+\bar v)\mathbf1$, implying $\bar v=\beta/(1-\beta)$. Since $\rho(\beta P)=\beta<1$, this is the unique finite solution. The Euler equation supplies the SDF, the Markov chain supplies conditional probabilities, and linear algebra solves the recursive pricing relation. If dividends differ from consumption, the ratio $d_{t+1}/d_t$ and the consumption-growth term in the SDF need not cancel under log utility.

In [ ]:
v_log = lucas_pd(P, g, beta, 1.0)
benchmark = beta/(1-beta)
print("Log-utility P/D ratios:", v_log)
print("Analytical benchmark:", benchmark)
print("Maximum absolute error:", np.max(np.abs(v_log-benchmark)))
assert np.allclose(P * g[None,:]**0, P)
assert np.allclose(v_log, benchmark*np.ones(n))